# Mini-TP 3 — Sirve tu modelo por gRPC

**Operaciones de Aprendizaje Automático II · CEIA · FIUBA · Entrega individual**

Expone el modelo de predicción de stroke (el mismo de la Sesión 1) como un servicio **gRPC**.

**Qué se entrega**
1. `scoring.proto` con un servicio para el modelo (mensajes de entrada y salida **tipados**).
2. Los *stubs* generados y un **servidor** que carga el modelo **una sola vez**.
3. Un **cliente** que llama al servicio (**unary**).
4. Un método de **server-streaming** que puntúa un lote.
5. Una **comparación de latencia** contra el endpoint REST de la Sesión 1, con reflexión.

## 1. El modelo

El mismo modelo de stroke reentrenado en `../common/train_model.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from common.inference import load_model

model_wrapper = load_model()
print(model_wrapper.name, "v" + str(model_wrapper.version), model_wrapper.metrics)

stroke_prediction_model_prod v1 {'accuracy': 0.9660668380462725, 'precision': 0.9758403361344538, 'recall': 0.9557613168724279, 'f1_score': 0.9656964656964657, 'train_observations': 7777, 'test_observations': 1945}


## 2. El contrato `.proto`

Mensajes tipados con las 10 features reales del modelo (no un `repeated double` genérico), un método unary (`Predict`) y uno de server-streaming (`PredictBatch`).

In [2]:
proto_source = '''
syntax = "proto3";
package scoring;

message StrokeFeatures {
  string gender = 1;
  double age = 2;
  int32 hypertension = 3;
  int32 heart_disease = 4;
  string ever_married = 5;
  string work_type = 6;
  string residence_type = 7;
  double avg_glucose_level = 8;
  double bmi = 9;
  string smoking_status = 10;
}

message StrokeBatch {
  repeated StrokeFeatures items = 1;
}

message Prediction {
  bool stroke_predicted = 1;
  string label = 2;
  double probability = 3;
  string model_name = 4;
  int32 model_version = 5;
}

service Scoring {
  rpc Predict(StrokeFeatures) returns (Prediction);
  rpc PredictBatch(StrokeBatch) returns (stream Prediction);
}
'''

with open("scoring.proto", "w") as f:
    f.write(proto_source)
print("scoring.proto escrito")

scoring.proto escrito


## 3. Genera los stubs

In [3]:
import subprocess, os

subprocess.run([
    sys.executable, "-m", "grpc_tools.protoc",
    "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto",
], check=True, cwd=".")
print("Generados:", [f for f in os.listdir(".") if f.startswith("scoring_pb2")])

Generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 4. El servidor

Carga el modelo una sola vez (ya cargado arriba) e implementa `Predict` y `PredictBatch`.

In [4]:
import grpc
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc

def _features_to_dict(f):
    return {
        "gender": f.gender, "age": f.age, "hypertension": f.hypertension,
        "heart_disease": f.heart_disease, "ever_married": f.ever_married,
        "work_type": f.work_type, "Residence_type": f.residence_type,
        "avg_glucose_level": f.avg_glucose_level, "bmi": f.bmi,
        "smoking_status": f.smoking_status,
    }

def _to_prediction(stroke_predicted, label, probability):
    return scoring_pb2.Prediction(
        stroke_predicted=stroke_predicted, label=label, probability=probability,
        model_name=model_wrapper.name, model_version=model_wrapper.version,
    )

class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    def Predict(self, request, context):
        stroke_predicted, label, probability = model_wrapper.predict_one(_features_to_dict(request))
        return _to_prediction(stroke_predicted, label, probability)

    def PredictBatch(self, request, context):
        features_list = [_features_to_dict(item) for item in request.items]
        for stroke_predicted, label, probability in model_wrapper.predict_batch(features_list):
            yield _to_prediction(stroke_predicted, label, probability)

servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50052")
servidor.start()
print("Servidor gRPC en localhost:50052")

Servidor gRPC en localhost:50052


## 5. El cliente (unary)

In [5]:
canal = grpc.insecure_channel("localhost:50052")
stub = scoring_pb2_grpc.ScoringStub(canal)

entrada = scoring_pb2.StrokeFeatures(
    gender="Male", age=67, hypertension=0, heart_disease=1, ever_married="Yes",
    work_type="Private", residence_type="Urban", avg_glucose_level=228.69,
    bmi=36.6, smoking_status="smokes",
)
r = stub.Predict(entrada)
print(r)

label: "Not likely to have a stroke"
probability: 0.15
model_name: "stroke_prediction_model_prod"
model_version: 1



## 6. Streaming

`PredictBatch` puntúa un lote y devuelve una predicción por paciente.

In [6]:
lote = scoring_pb2.StrokeBatch(items=[
    entrada,
    scoring_pb2.StrokeFeatures(gender="Female", age=30, hypertension=0, heart_disease=0,
                               ever_married="No", work_type="Private", residence_type="Urban",
                               avg_glucose_level=90.0, bmi=22.0, smoking_status="never smoked"),
    scoring_pb2.StrokeFeatures(gender="Male", age=80, hypertension=1, heart_disease=1,
                               ever_married="Yes", work_type="Private", residence_type="Rural",
                               avg_glucose_level=250.0, bmi=40.0, smoking_status="smokes"),
])
for p in stub.PredictBatch(lote):
    print(round(p.probability, 3), "-", p.label)

0.15 - Not likely to have a stroke
0.0 - Not likely to have a stroke
0.45 - Not likely to have a stroke


## 7. Comparación de latencia gRPC vs REST

Se levanta también la API REST de la Sesión 1 en un hilo, con `requests.Session()` (keep-alive) para que la comparación no quede dominada por el costo de abrir una conexión TCP nueva en cada request.

In [7]:
import threading, time, uvicorn, requests

sys.path.insert(0, str(Path.cwd().parent / "mini_tp1_rest"))
from api import app as rest_app

def _run_rest():
    uvicorn.Server(uvicorn.Config(rest_app, host="127.0.0.1", port=8012, log_level="warning")).run()

threading.Thread(target=_run_rest, daemon=True).start(); time.sleep(2)

rest_case = {
    "gender": "Male", "age": 67, "hypertension": 0, "heart_disease": 1, "ever_married": "Yes",
    "work_type": "Private", "Residence_type": "Urban", "avg_glucose_level": 228.69,
    "bmi": 36.6, "smoking_status": "smokes",
}
REST_URL = "http://127.0.0.1:8012/v1/predict"
N = 200

stub.Predict(entrada)  # warm-up
t = time.perf_counter()
for _ in range(N):
    stub.Predict(entrada)
grpc_ms = (time.perf_counter() - t) / N * 1000

with requests.Session() as session:
    session.post(REST_URL, json=rest_case)  # warm-up
    t = time.perf_counter()
    for _ in range(N):
        session.post(REST_URL, json=rest_case)
    rest_ms = (time.perf_counter() - t) / N * 1000

print(f"gRPC: {grpc_ms:.3f} ms/llamada")
print(f"REST: {rest_ms:.3f} ms/llamada")
print(f"gRPC fue {rest_ms / grpc_ms:.2f}x mas rapido que REST en este caso.")

gRPC: 10.531 ms/llamada
REST: 14.321 ms/llamada
gRPC fue 1.36x mas rapido que REST en este caso.


### Reflexión

Con un solo modelo chico y llamadas locales, gRPC y REST rinden parecido una vez que ambos reusan la conexión (la diferencia queda dominada por el tiempo de inferencia del modelo, no por el protocolo). La ventaja real de gRPC aparece en comunicación **interna** entre microservicios de alta frecuencia (HTTP/2 multiplexado, payload binario más chico que JSON, contrato `.proto` estricto) y en el **streaming nativo** para lotes, sin reabrir conexión por cada item. Usaría REST/GraphQL en el borde público de la plataforma (lo habla cualquier cliente sin tooling extra) y gRPC para la comunicación interna entre el servicio de scoring y otros microservicios. El costo de gRPC es el tooling (generar y versionar stubs a partir del `.proto`) y que no lo consume un navegador sin un proxy.

In [8]:
canal.close(); servidor.stop(0)
print("Servidor detenido.")

Servidor detenido.
